FIPVで811と814(RRXRRSのモチーフ内)に変異を持たない配列のlist

In [13]:
import pandas as pd
from Bio import SeqIO
import os

# --- 1. 設定 ---
input_fasta = "/Users/izumikida/FCoV_seq_relatives/FCoV1_S_final/FCoV1_S_all_seq_cleaned_cds_linsi_curated_fullllength_fix.aa_dates.fasta"
# コンセンサス配列のパス（同じディレクトリにあると想定、なければフルパスに変更してください）
consensus_fasta = "/Users/izumikida/FCoV_seq_relatives/FCoV1_S_final/snipit_NRR_number/fecv_consensus.fas" 
output_csv = "phen_scores_dates.csv"

# 調査したいポジション (1-based index)
positions = [791, 809, 811, 813, 814 ,818, 826, 1056, 1072, 1074, 1078, 1129, 1133, 1135, 1160, 1212, 1363, 1368]

# --- 2. コンセンサス配列の読み込み ---
if not os.path.exists(consensus_fasta):
    print(f"エラー: コンセンサスファイルが見つかりません: {consensus_fasta}")
    exit()

consensus_record = next(SeqIO.parse(consensus_fasta, "fasta"))
consensus_seq = str(consensus_record.seq)

# --- 3. 各配列の比較とスコアリング ---
results = []

if not os.path.exists(input_fasta):
    print(f"エラー: 入力ファイルが見つかりません: {input_fasta}")
else:
    for record in SeqIO.parse(input_fasta, "fasta"):
        sample_id = record.id.split('|')[0]
        sample_seq = str(record.seq)
        
        # サンプルごとのスコアを格納する辞書
        score_row = {'Accession': sample_id}
        
        for pos in positions:
            idx = pos - 1  # 0-based indexに変換
            
            # 配列の長さチェック
            if idx < len(sample_seq) and idx < len(consensus_seq):
                ref_aa = consensus_seq[idx]
                sample_aa = sample_seq[idx]
                
                # コンセンサスと同じなら0、変異（ギャップ含む）があれば1
                if sample_aa == ref_aa:
                    score_row[f"SNP_{pos}"] = 0
                else:
                    score_row[f"SNP_{pos}"] = 1
            else:
                score_row[f"SNP_{pos}"] = None # 配列が足りない場合
        
        results.append(score_row)

# --- 4. CSVとして保存 ---
df_result = pd.DataFrame(results)
df_result.set_index('Accession', inplace=True)
df_result.to_csv(output_csv)

print(f"成功: {len(df_result)} サンプルのスコアを {output_csv} に保存しました。")

成功: 120 サンプルのスコアを phen_scores_dates.csv に保存しました。


In [4]:
import pandas as pd
from Bio import SeqIO
import os

# --- 1. データの読み込み ---
# 参照ファイルを phen_scores_260318.csv に変更
df_snps_raw = pd.read_csv('/Users/izumikida/FCoV_seq_relatives/treeWAS/phen_scores_260318.csv', index_col=0)
df_meta = pd.read_csv('/Users/izumikida/FCoV_seq_relatives/FCoV1_S_final/FCoV1_S_all_seq_final_meta_20260306.tsv', sep='\t')

# --- 2. 条件フィルタリング ---
# SNP_ で始まるカラムのみを抽出し、数値化
snp_cols = [c for c in df_snps_raw.columns if c.startswith('SNP_')]
df_snps = df_snps_raw[snp_cols].copy()
df_snps.columns = [int(c.replace('SNP_', '')) for c in df_snps.columns]

# 条件A: 811, 813, 814 がすべて 0
# (もし phen_scores に 813 が含まれていない場合はエラーになるため、存在確認を入れるとより安全です)
condition_snp = (df_snps[811] == 0) & (df_snps[813] == 0) & (df_snps[814] == 0)

# 条件B: メタデータで biotype が FIPV
fipv_ids = set(df_meta[df_meta['biotype'] == 'FIPV']['Accession'])

# 両方の条件を満たすIDを抽出
target_df = df_snps[condition_snp & df_snps.index.isin(fipv_ids)]
target_accessions = target_df.index.tolist()

print(f"抽出条件 (811=0, 813=0, 814=0 & FIPV) に一致したAccession数: {len(target_accessions)}")

# --- 3. スコアテーブル（CSV）の出力 ---
output_table_csv = "FIPV_811_813_814_zero_snps.csv"
target_df.to_csv(output_table_csv)
print(f"成功: SNPスコアテーブルを {output_table_csv} に保存しました。")

# --- 4. FASTAファイルからの抽出 ---
input_fasta = "/Users/izumikida/FCoV_seq_relatives/FCoV1_S_final/FCoV1_S_all_seq_cleaned_cds_linsi_curated_fullllength_fix.aa_dates.fasta"
output_fasta = "FIPV_811_813_814_zero_extracted.fasta"

if not os.path.exists(input_fasta):
    print(f"エラー: 指定されたパスにファイルが見わたりません: {input_fasta}")
else:
    extracted_records = []
    for record in SeqIO.parse(input_fasta, "fasta"):
        # FASTAヘッダーのID部分（パイプ区切り等）と照合
        record_id_clean = record.id.split('|')[0]
        if record_id_clean in target_accessions:
            extracted_records.append(record)

    if extracted_records:
        SeqIO.write(extracted_records, output_fasta, "fasta")
        print(f"成功: {len(extracted_records)} 件の配列を {output_fasta} に保存しました。")
    else:
        print("警告: 条件に一致する配列がFASTAファイル内に見つかりませんでした。")

抽出条件 (811=0, 813=0, 814=0 & FIPV) に一致したAccession数: 20
成功: SNPスコアテーブルを FIPV_811_813_814_zero_snps.csv に保存しました。
成功: 20 件の配列を FIPV_811_813_814_zero_extracted.fasta に保存しました。


818はFCSの後ろ、gapを便宜上詰めたためアライメントの真偽不明。fecv_concensusと比べるとfecvでもfipvでも変異あり（score = 1）のため、今回のFisherの結果からはのぞく。

In [14]:
import pandas as pd

# --- 1. データの読み込み ---
# スコアデータとメタデータの読み込み
df_scores = pd.read_csv('/Users/izumikida/FCoV_seq_relatives/treeWAS/phen_scores_dates.csv', index_col=0)
df_meta = pd.read_csv('/Users/izumikida/FCoV_seq_relatives/FCoV1_S_final/FCoV1_S_all_seq_final_meta_20260306.tsv', sep='\t')

# --- 2. データの紐付けと抽出 ---
# メタデータから FECV の Accession リストを作成
fecv_accessions = df_meta[df_meta['biotype'] == 'FECV']['Accession'].tolist()

# スコアデータの中から FECV に該当する行のみを抽出
df_fecv = df_scores[df_scores.index.isin(fecv_accessions)].copy()

# --- 3. ポジション818のスコアを確認 ---
column_name = 'SNP_818'

if column_name in df_fecv.columns:
    # 必要な列（SNP_818）だけを選択
    result_table = df_fecv[[column_name]]
    
    # 統計情報の表示
    print(f"--- FECVサンプルの {column_name} スコア統計 ---")
    print(result_table[column_name].value_counts())

--- FECVサンプルの SNP_818 スコア統計 ---
SNP_818
1    41
0     9
Name: count, dtype: int64


In [15]:
import pandas as pd

# 1. 元のファイルを読み込み
input_path = '/Users/izumikida/FCoV_seq_relatives/treeWAS/phen_scores_dates.csv'
df = pd.read_csv(input_path, index_col=0)

# 2. 'SNP_818' カラムを削除
# errors='ignore' をつけることで、万が一カラムが存在しなくてもエラーにならずに進みます
column_to_drop = 'SNP_818'
df_dropped = df.drop(columns=[column_to_drop], errors='ignore')

# 3. 新しいファイル名で出力
output_path = '/Users/izumikida/FCoV_seq_relatives/treeWAS/phen_scores_dates_rm_818.csv'
df_dropped.to_csv(output_path)

print(f"成功: '{column_to_drop}' を削除したファイルを以下に保存しました:")
print(output_path)
print(f"現在のカラム数: {len(df_dropped.columns)} (元: {len(df.columns)})")

成功: 'SNP_818' を削除したファイルを以下に保存しました:
/Users/izumikida/FCoV_seq_relatives/treeWAS/phen_scores_dates_rm_818.csv
現在のカラム数: 17 (元: 18)


treeWASで以前作成したtreeWAS_phen.csvをfinal datasetと揃えるため、dates不明とoutlierの配列を削除する。

In [1]:
list_rm = ['DQ848678.1','D32044.1','AB088222.1', 'c57']

In [8]:
import pandas as pd
meta2 = pd.read_csv("/Users/izumikida/FCoV_seq_relatives/treeWAS/prelim/treeWAS_phen.csv")
meta2

,Accession,biotype
0,PQ416775.1,1
1,PX315748.1,1
2,PX315749.1,1
3,PX315750.1,1
4,PX315751.1,1
...,...,...
119,AB088222.1,1
120,FJ917522.1,0
121,FJ917535.1,1
122,FJ917523.1,1


In [7]:
meta2_filtered = meta2[~meta2["Accession"].isin(list＿rm)].copy()
meta2_filtered

,Accession,biotype
0,PQ416775.1,1
1,PX315748.1,1
2,PX315749.1,1
3,PX315750.1,1
4,PX315751.1,1
...,...,...
118,FJ917521.1,1
120,FJ917522.1,0
121,FJ917535.1,1
122,FJ917523.1,1


In [12]:
meta2_filtered.to_csv('treeWAS_phen_dates.csv', index=False)